# Mutual Fund Analytics Platform — Day 4: Performance Analytics
**Bluestock Fintech Capstone Project**  
Author: Shahin Shafi | June 2026

This notebook computes risk & return metrics from raw NAV data:
Daily Returns, CAGR, Sharpe Ratio, Sortino Ratio, Alpha, Beta, Maximum Drawdown, Fund Scorecard, and Benchmark Comparison.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from IPython.display import Image, display

BASE_DIR   = Path('.')
DB_PATH    = BASE_DIR / 'bluestock_mf.db'
PROC_DIR   = BASE_DIR / 'data' / 'processed'
CHARTS_DIR = BASE_DIR / 'reports' / 'charts'

RF_ANNUAL    = 0.065
RF_DAILY     = RF_ANNUAL / 252
TRADING_DAYS = 252

conn = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)

## Load Data

In [ ]:
df_nav = pd.read_sql("""
    SELECT n.amfi_code, n.date_id, n.nav,
           f.scheme_name, f.sub_category, f.category,
           f.plan, f.expense_ratio_pct, f.fund_house,
           COALESCE(f.benchmark_key, 'NIFTY100') AS benchmark_key
    FROM fact_nav n JOIN dim_fund f ON n.amfi_code=f.amfi_code
""", conn)
df_nav['date'] = pd.to_datetime(df_nav['date_id'])
df_nav = df_nav.sort_values(['amfi_code','date']).reset_index(drop=True)

df_bench = pd.read_sql('SELECT date_id, index_name, close_value FROM benchmark_indices', conn)
df_bench['date'] = pd.to_datetime(df_bench['date_id'])

df_fund = pd.read_sql('SELECT * FROM dim_fund', conn)

nav_wide = df_nav.pivot_table(index='date', columns='amfi_code', values='nav')
returns  = nav_wide.pct_change().dropna()
print(f'NAV matrix : {nav_wide.shape}')
print(f'Returns    : {returns.shape}')

## Step 1: Daily Returns — Distribution
Formula: `daily_return = (NAV_t / NAV_t-1) - 1`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of all daily returns
all_ret = returns.values.flatten()
all_ret = all_ret[~np.isnan(all_ret)]
axes[0].hist(all_ret * 100, bins=100, color='#1565C0', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of All Daily Returns (All 40 Funds)', fontsize=12)
axes[0].set_xlim(-8, 8)

# Rolling 30-day volatility for a sample fund
sample = returns.iloc[:, 0]
rolling_vol = sample.rolling(30).std() * np.sqrt(252) * 100
axes[1].plot(rolling_vol.index, rolling_vol, color='#E53935', linewidth=1.5)
axes[1].set_title(f'Rolling 30-Day Annualised Volatility', fontsize=12)
axes[1].set_ylabel('Volatility (%)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(CHARTS_DIR / '17_returns_distribution.png', dpi=150)
plt.show()
print(f'Mean: {all_ret.mean()*100:.4f}%  Std: {all_ret.std()*100:.4f}%')
print(f'Min: {all_ret.min()*100:.2f}%   Max: {all_ret.max()*100:.2f}%')

## Step 2: CAGR — 1yr, 3yr, 5yr
Formula: `CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

In [ ]:
scorecard = pd.read_csv(PROC_DIR / 'fund_scorecard.csv')
cagr_cols = ['scheme_name','sub_category','plan','cagr_1yr_pct','cagr_3yr_pct','cagr_5yr_pct']
print('Top 10 Funds by 3-Year CAGR:')
print(scorecard.nlargest(10,'cagr_3yr_pct')[cagr_cols].to_string(index=False))

## Step 3 & 4: Sharpe & Sortino Ratios
- **Sharpe** = `(Rp - Rf) / Std(Rp) * sqrt(252)` — penalises total volatility
- **Sortino** = `(Rp - Rf) / Downside_Std * sqrt(252)` — only penalises bad volatility
- Rf = 6.5% (RBI repo rate)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_data = scorecard.nlargest(20, 'sharpe_ratio').copy()
plot_data['short_name'] = plot_data['scheme_name'].apply(lambda x: str(x).split(' - ')[0][:20])
x = np.arange(len(plot_data))
ax.bar(x - 0.2, plot_data['sharpe_ratio'],  0.4, label='Sharpe',  color='#1565C0', alpha=0.85)
ax.bar(x + 0.2, plot_data['sortino_ratio'], 0.4, label='Sortino', color='#2E7D32', alpha=0.85)
ax.axhline(1.0, color='red', linewidth=1.2, linestyle='--', alpha=0.7, label='Sharpe=1 threshold')
ax.set_xticks(x)
ax.set_xticklabels(plot_data['short_name'], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Ratio')
ax.set_title('Sharpe vs Sortino Ratio — Top 20 Funds', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(CHARTS_DIR / '18_sharpe_sortino.png', dpi=150)
plt.show()

## Step 5: Alpha & Beta
- **Beta** = slope of OLS regression (fund returns vs benchmark returns)
- **Alpha** = intercept × 252 (annualised excess return above what Beta predicts)
- Using `scipy.stats.linregress`

In [ ]:
ab = pd.read_csv(PROC_DIR / 'alpha_beta.csv')

fig, ax = plt.subplots(figsize=(10, 7))
cats = ab['sub_category'].unique()
colors = ['#1565C0','#E53935','#2E7D32','#6A1B9A','#E65100','#795548']
for i, cat in enumerate(cats):
    sub = ab[ab['sub_category']==cat]
    ax.scatter(sub['beta'], sub['alpha_ann_pct'],
               label=cat, s=100, alpha=0.85,
               color=colors[i % len(colors)], edgecolors='white')
    for _, row in sub.iterrows():
        ax.annotate(str(row['scheme_name']).split(' - ')[0][:14],
                    (row['beta'], row['alpha_ann_pct']),
                    textcoords='offset points', xytext=(4,3), fontsize=6.5, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(1, color='gray',  linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_xlabel('Beta (market sensitivity)')
ax.set_ylabel('Alpha (annualised %)')
ax.set_title('Alpha vs Beta — All 40 Funds', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(CHARTS_DIR / '19_alpha_beta.png', dpi=150)
plt.show()

print('Top 5 by Alpha:')
print(ab.nlargest(5,'alpha_ann_pct')[['scheme_name','alpha_ann_pct','beta','r_squared']].to_string(index=False))

## Step 6: Maximum Drawdown
Formula: `max_dd = min(NAV_t / rolling_max(NAV) - 1)`
Finds the worst peak-to-trough loss for each fund.

In [ ]:
dd_data = scorecard[['scheme_name','max_drawdown_pct','sub_category']].copy()
dd_data['short'] = dd_data['scheme_name'].apply(lambda x: str(x).split(' - ')[0][:22])
dd_sorted = dd_data.sort_values('max_drawdown_pct')

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#E53935' if v < -20 else '#FF9800' if v < -10 else '#4CAF50'
          for v in dd_sorted['max_drawdown_pct']]
ax.barh(dd_sorted['short'], dd_sorted['max_drawdown_pct'],
        color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Maximum Drawdown (%)')
ax.set_title('Maximum Drawdown by Fund\n(Red = >20% loss, Orange = 10-20%, Green = <10%)', fontsize=13)
plt.tight_layout()
plt.savefig(CHARTS_DIR / '20_max_drawdown.png', dpi=150)
plt.show()

## Step 7: Fund Scorecard (0-100 Composite)
Weights: 30% × 3yr CAGR + 25% × Sharpe + 20% × Alpha + 15% × Expense (inverse) + 10% × Max DD (inverse)

In [ ]:
top20 = scorecard.head(20).copy()
top20['short'] = top20['scheme_name'].apply(lambda x: str(x).split(' - ')[0][:24])

fig, ax = plt.subplots(figsize=(12, 8))
bar_colors = ['#1565C0' if p=='Direct' else '#90CAF9'
              for p in top20['plan']]
bars = ax.barh(top20['short'][::-1], top20['composite_score'][::-1],
               color=bar_colors[::-1], edgecolor='white')
for bar, score in zip(bars, top20['composite_score'][::-1]):
    ax.text(score + 0.3, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=8)
ax.set_xlabel('Composite Score (0-100)')
ax.set_title('Fund Scorecard — Top 20 Funds\n(Dark=Direct, Light=Regular)', fontsize=13)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig(CHARTS_DIR / '21_fund_scorecard.png', dpi=150)
plt.show()

print('Full scorecard saved to: data/processed/fund_scorecard.csv')

## Step 8: Benchmark Comparison Chart

In [ ]:
display(Image(filename=str(CHARTS_DIR / '16_benchmark_comparison.png'), width=900))

In [ ]:
conn.close()
charts = list(CHARTS_DIR.glob('*.png'))
print(f'Day 4 Complete!')
print(f'Total charts: {len(charts)}')
print('Outputs: fund_scorecard.csv, alpha_beta.csv, cagr_metrics.csv')